In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from mimiciii_db import DB
from mimiciii_db.config import db_url

- icu stays 24 hrs admission
- age >=16
- elective vs nonelective
- patients, admissions, icustays

labevents, chartevents

sofa:
- respiratory system PaO2/FiO2
- coagulation platelet count
- liver bilirubin level
- cardiovascular blood pressure
- central nervous system
- renal

each 4 points, sum of 24

In [ ]:
db = DB.from_url(db_url())


In [ ]:
icustays = db.table_df("icustays", schema="mimiciii")
icustays

In [ ]:
patients = db.table_df("patients", schema="mimiciii")
patients.head()

In [ ]:
admissions = db.table_df("admissions", schema="mimiciii")
admissions.head()

In [ ]:
# set your schema name here
SCHEMA = "mimiciii"

from sqlalchemy import text

def exec_sql(db, sql: str, params=None):
    """Run DDL/CTE/INSERT… statements that don't return a DataFrame."""
    with db.engine.begin() as conn:
        conn.execute(text(sql), params or {})


In [ ]:
exec_sql(db, f"""
DROP TABLE IF EXISTS tmp_first_icu;
CREATE TEMP TABLE tmp_first_icu AS
WITH first_icu AS (
  SELECT
    icustay_id,
    hadm_id,
    subject_id,
    intime,
    ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY intime) AS rn
  FROM {SCHEMA}.icustays
),
first_only AS (
  SELECT
    icustay_id,
    hadm_id,
    subject_id,
    intime
  FROM first_icu
  WHERE rn = 1
),
adult AS (
  SELECT
    f.icustay_id,
    f.hadm_id,
    f.subject_id,
    f.intime,
    adm.admittime,
    pat.dob,
    EXTRACT(EPOCH FROM (adm.admittime - pat.dob))/86400.0/365.2425 AS age
  FROM first_only f
  JOIN {SCHEMA}.admissions adm
    ON adm.hadm_id = f.hadm_id
  JOIN {SCHEMA}.patients pat
    ON pat.subject_id = f.subject_id
)
SELECT *
FROM adult
WHERE age >= 16;
""")


In [ ]:
db.query_df("SELECT COUNT(*) AS n, MIN(age) AS min_age, MAX(age) AS max_age FROM tmp_first_icu")


In [ ]:
exec_sql(db, f"""
DROP TABLE IF EXISTS tmp_ce24;
CREATE TEMP TABLE tmp_ce24 AS
SELECT ce.icustay_id, ce.charttime, ce.itemid, ce.valuenum
FROM {SCHEMA}.chartevents ce
JOIN tmp_first_icu fi USING (icustay_id)
WHERE ce.charttime>=fi.intime
  AND ce.charttime< fi.intime + INTERVAL '24 hours'
  AND ce.valuenum IS NOT NULL
  AND ce.error IS DISTINCT FROM 1;

DROP TABLE IF EXISTS tmp_le24;
CREATE TEMP TABLE tmp_le24 AS
SELECT fi.icustay_id, le.charttime, le.itemid, le.valuenum
FROM {SCHEMA}.labevents le
JOIN tmp_first_icu fi USING (hadm_id)
WHERE le.charttime>=fi.intime
  AND le.charttime< fi.intime + INTERVAL '24 hours'
  AND le.valuenum IS NOT NULL
  AND le.flag IS DISTINCT FROM 'ERROR';

DROP TABLE IF EXISTS tmp_oe24;
CREATE TEMP TABLE tmp_oe24 AS
SELECT fi.icustay_id, oe.charttime, oe.value AS urine_cc
FROM {SCHEMA}.outputevents oe
JOIN tmp_first_icu fi USING (icustay_id, hadm_id)
WHERE oe.charttime>=fi.intime
  AND oe.charttime< fi.intime + INTERVAL '24 hours'
  AND oe.value IS NOT NULL;
""")

# row counts to sanity check
db.query_df("""
SELECT
  (SELECT COUNT(*) FROM tmp_ce24) AS ce_rows,
  (SELECT COUNT(*) FROM tmp_le24) AS le_rows,
  (SELECT COUNT(*) FROM tmp_oe24) AS oe_rows
""")
